# OntoNotes5 Olfactory-NER Experiments

**Dataset**: OntoNotes5 (18 entity types vs 4 in CoNLL-2003)

**Experiments**:
1. Baseline BiLSTM-CRF
2. Olfactory (GELU) - full model
3. Olfactory (GELU) - no glomeruli

**This notebook is STANDALONE** - run on fresh Colab session!

## 1. Setup

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repository
!git clone https://github.com/bhushan1729/olfaction-inspired-ner.git
%cd olfaction-inspired-ner

In [ ]:
# Install dependencies
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard datasets requests

In [ ]:
# Download GloVe
import os
if not os.path.exists('./data/glove.6B.300d.txt'):
    print("Downloading GloVe...")
    !mkdir -p data
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip
    !unzip -q data/glove.6B.zip -d data/
    !rm data/glove.6B.zip
    print("✓ Done")
else:
    print("✓ GloVe exists")

## 2. Experiment 1: Baseline

In [ ]:
# Train baseline BiLSTM-CRF
!python src/train_ontonotes.py \
  --config config/ontonotes_experiments.yaml \
  --experiment baseline_ontonotes \
  --save_dir results/ontonotes/baseline

## 3. Experiment 2: Olfactory (GELU)

In [ ]:
# Train olfactory with GELU activation
!python src/train_ontonotes.py \
  --config config/ontonotes_experiments.yaml \
  --experiment olfactory_gelu_ontonotes \
  --save_dir results/ontonotes/olfactory_gelu

## 4. Experiment 3: Olfactory (GELU, No Glomeruli)

In [ ]:
# Train olfactory without glomeruli (best variant from CoNLL-2003)
!python src/train_ontonotes.py \
  --config config/ontonotes_experiments.yaml \
  --experiment olfactory_no_glomeruli_ontonotes \
  --save_dir results/ontonotes/olfactory_no_glomeruli

## 5. Compare Results

In [ ]:
import json
import pandas as pd

# Load all results
results = {}
for name, path in [
    ('Baseline', 'results/ontonotes/baseline/results.json'),
    ('Olfactory (GELU)', 'results/ontonotes/olfactory_gelu/results.json'),
    ('Olfactory (No Glom)', 'results/ontonotes/olfactory_no_glomeruli/results.json')
]:
    with open(path) as f:
        results[name] = json.load(f)

# Create comparison table
comparison = []
for name, res in results.items():
    comparison.append({
        'Model': name,
        'Test F1': f"{res['test']['f1']:.4f}",
        'Precision': f"{res['test']['precision']:.4f}",
        'Recall': f"{res['test']['recall']:.4f}"
    })

df = pd.DataFrame(comparison)
print("\n" + "="*60)
print(" "*15 + "OntoNotes5 Results")
print("="*60)
print(df.to_string(index=False))
print("="*60)

In [ ]:
# Show per-entity F1 for all models
print("\nPer-Entity F1 Scores:\n")
for name, res in results.items():
    print(f"\n{name}:")
    print("-" * 40)
    entities = res['test']['per_entity']
    for entity in sorted(entities.keys()):
        if entity not in ['micro avg', 'macro avg', 'weighted avg']:
            print(f"  {entity:<12} {entities[entity]:.4f}")

## 6. Download Results

In [ ]:
# Package all results
!zip -r ontonotes_results.zip results/ontonotes/

from google.colab import files
files.download('ontonotes_results.zip')

print("\n✓ Results downloaded!")

---

## Summary

You've now trained and evaluated olfactory-NER on **OntoNotes5**:
- 18 entity types (vs 4 in CoNLL-2003)
- More challenging dataset
- Tests generalization

**Compare with CoNLL-2003 results to see how well the models generalize!**